# 01 · Data — เตรียม ThaiSum
**เป้าหมาย:** ทำความสะอาด → สุ่มชุดย่อย → แบ่ง train/val/test → บันทึกให้ทุกคนใช้ร่วมกัน → สรุปสถิติความยาว

ผลลัพธ์ (ทุกคนในทีมโหลดจาก `data/processed/` เท่านั้น ห้ามสุ่ม/แบ่งเอง เพื่อให้ผลเทียบกันได้):
- `train.jsonl`, `val.jsonl`, `test.jsonl`
- `manifest.json` (config, จำนวนแถว, SHA-256 ของแต่ละไฟล์)
- `length_stats.csv` + กราฟใน `reports/`

ThaiSum มีคอลัมน์ `title, body, summary, type, tags, url` (ข่าวจาก Thairath, ThaiPBS, Prachathai, The Standard)

In [ ]:
%pip install -q pandas numpy matplotlib pythainlp datasets pyarrow
import hashlib, json, re, unicodedata
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ---------- CONFIG (แก้ที่นี่ที่เดียว) ----------
SEED        = 42
N_SUBSET    = 50_000          # จำนวนบทความหลังสุ่ม (None = ใช้ทั้งหมด)
SPLIT       = (0.8, 0.1, 0.1) # train / val / test
MIN_BODY_W  = 150             # เกณฑ์เดียวกับ ThaiSum: body >= 150 คำ
MIN_SUM_W   = 15              # summary >= 15 คำ
MAX_BODY_W  = 3000            # ตัดข่าวยาวผิดปกติ
MAX_ABSTR   = 0.60            # unigram abstractedness สูงสุด
SOURCE      = "hf"            # "hf" = HuggingFace nakhun/thaisum, "csv" = ไฟล์ thaisum.csv ในเครื่อง
CSV_PATH    = "data/raw/thaisum.csv"

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RAW, OUT, REP = ROOT/"data/raw", ROOT/"data/processed", ROOT/"reports"
for p in (RAW, OUT, REP): p.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

## 1. โหลดข้อมูลดิบ

In [ ]:
if SOURCE == "hf":
    from datasets import load_dataset
    try:
        # ทางที่ 1: branch parquet ที่ HF แปลงให้อัตโนมัติ (ไม่ต้องใช้สคริปต์)
        ds = load_dataset("nakhun/thaisum", revision="refs/convert/parquet")
    except Exception as e:
        print("parquet branch failed:", e)
        # ทางที่ 2: ต้องลง datasets เวอร์ชันเก่า แล้ว restart runtime ก่อนรัน
        #   !pip install -q "datasets<4"
        ds = load_dataset("nakhun/thaisum", trust_remote_code=True)
    raw = pd.concat([ds[k].to_pandas() for k in ds.keys()], ignore_index=True)
else:
    raw = pd.read_csv(ROOT/CSV_PATH)

print(raw.columns.tolist(), raw.shape)
raw = raw[["title","body","summary","type","tags","url"]]
raw.head(3)


## 2. ทำความสะอาด

In [ ]:
from pythainlp.tokenize import word_tokenize

TAG_RE   = re.compile(r"<[^>]+>")
URL_RE   = re.compile(r"https?://\S+|www\.\S+")
WS_RE    = re.compile(r"\s+")
ZW_RE    = re.compile(r"[\u200b\u200c\u200d\ufeff]")   # zero-width chars

def clean(t):
    if not isinstance(t, str): return ""
    t = unicodedata.normalize("NFC", t)
    t = ZW_RE.sub("", t)
    t = TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    return WS_RE.sub(" ", t).strip()

df = raw.copy()
log = {"raw": len(df)}

for c in ("title","body","summary"):
    df[c] = df[c].map(clean)

df = df[(df.body != "") & (df.summary != "")];                    log["non_empty"] = len(df)
df = df.drop_duplicates(subset="url");                            log["dedup_url"] = len(df)
df = df.drop_duplicates(subset="body");                           log["dedup_body"] = len(df)
df = df[df.body != df.summary];                                   log["body!=summary"] = len(df)

In [ ]:
# นับคำด้วย newmm (ช้าสำหรับข้อมูลทั้งชุด -> ใช้ multiprocessing หรือสุ่มก่อนถ้าเครื่องช้า)
def toks(t): return [w for w in word_tokenize(t, engine="newmm", keep_whitespace=False)]

# เพื่อลดเวลา: สุ่มเผื่อ 1.5 เท่าก่อน แล้วค่อยกรองด้วยความยาว/abstractedness
if N_SUBSET and len(df) > int(N_SUBSET*1.5):
    df = df.sample(n=int(N_SUBSET*1.5), random_state=SEED)
df = df.reset_index(drop=True)

df["body_tok"] = df.body.map(toks)
df["sum_tok"]  = df.summary.map(toks)
df["body_w"], df["sum_w"] = df.body_tok.map(len), df.sum_tok.map(len)

def abstractedness(b, s):
    s = [w for w in s]; 
    return sum(w not in b for w in s)/max(len(s),1)
bset = df.body_tok.map(set)
df["abstr1"] = [abstractedness(b, s) for b, s in zip(bset, df.sum_tok)]

m = (df.body_w.between(MIN_BODY_W, MAX_BODY_W)) & (df.sum_w >= MIN_SUM_W) & (df.abstr1 <= MAX_ABSTR) \
    & (df.sum_w < df.body_w)
df = df[m].reset_index(drop=True); log["length+abstr_filter"] = len(df)
pd.Series(log, name="rows").to_frame()

## 3. สุ่มชุดย่อย (stratified ตาม `type`)

In [ ]:
df["type"] = df["type"].fillna("unknown")
if N_SUBSET and len(df) > N_SUBSET:
    frac = N_SUBSET/len(df)
    df = (df.groupby("type", group_keys=False)
            .apply(lambda g: g.sample(frac=frac, random_state=SEED))
            .reset_index(drop=True))
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(len(df)); df["type"].value_counts().head(10)

## 4. แบ่ง train / val / test (stratified, fix seed)

In [ ]:
assert abs(sum(SPLIT)-1) < 1e-9
df["split"] = ""
for t, g in df.groupby("type"):
    idx = g.index.to_numpy().copy(); rng.shuffle(idx)
    n = len(idx); n_tr = int(n*SPLIT[0]); n_va = int(n*SPLIT[1])
    df.loc[idx[:n_tr], "split"] = "train"
    df.loc[idx[n_tr:n_tr+n_va], "split"] = "val"
    df.loc[idx[n_tr+n_va:], "split"] = "test"
df["id"] = [f"ts{i:06d}" for i in range(len(df))]

# ตรวจ leakage
S = {k:set(df[df.split==k].url) for k in ("train","val","test")}
assert not (S["train"]&S["val"] or S["train"]&S["test"] or S["val"]&S["test"])
df.split.value_counts()

## 5. บันทึกไฟล์ให้ทุกคน + manifest

In [ ]:
COLS = ["id","title","body","summary","type","tags","url","body_w","sum_w"]
def sha(p):
    h = hashlib.sha256()
    with open(p,"rb") as f:
        for ch in iter(lambda: f.read(1<<20), b""): h.update(ch)
    return h.hexdigest()

manifest = {"seed":SEED,"n_subset":N_SUBSET,"split":SPLIT,
            "filters":{"min_body_w":MIN_BODY_W,"max_body_w":MAX_BODY_W,"min_sum_w":MIN_SUM_W,"max_abstr1":MAX_ABSTR},
            "tokenizer":"pythainlp newmm","cleaning_log":log,"files":{}}
for k in ("train","val","test"):
    p = OUT/f"{k}.jsonl"
    df[df.split==k][COLS].to_json(p, orient="records", lines=True, force_ascii=False)
    manifest["files"][p.name] = {"rows": int((df.split==k).sum()), "sha256": sha(p)}
json.dump(manifest, open(OUT/"manifest.json","w"), ensure_ascii=False, indent=2)
manifest["files"]

**วิธีแจก:** commit `manifest.json` เข้า git เสมอ ส่วนไฟล์ `.jsonl` ถ้าใหญ่ให้ใช้ Git LFS / Drive / HF dataset แล้วให้ทุกคนตรวจ SHA-256 กับ manifest ก่อนใช้
```python
df = pd.read_json("data/processed/train.jsonl", lines=True)
```

## 6. สถิติความยาว

In [ ]:
d = df[df.split!=""].copy()
d["body_c"], d["sum_c"] = d.body.str.len(), d.summary.str.len()
d["ratio"] = d.sum_w/d.body_w
def desc(s): return s.describe(percentiles=[.5,.9,.95,.99]).round(1)
stats = pd.concat({
    "body_words": desc(d.body_w), "summary_words": desc(d.sum_w),
    "body_chars": desc(d.body_c), "summary_chars": desc(d.sum_c),
    "compression(sum/body)": desc(d.ratio)}, axis=1).T
stats.to_csv(REP/"length_stats.csv"); stats

In [ ]:
by_split = d.groupby("split")[["body_w","sum_w"]].agg(["mean","median","max"]).round(1)
by_type  = d.groupby("type")[["body_w","sum_w"]].agg(["count","mean"]).round(1).sort_values(("body_w","count"), ascending=False)
display(by_split); display(by_type.head(10))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15,3.8))
ax[0].hist(d.body_w.clip(upper=d.body_w.quantile(.99)), bins=60); ax[0].set_title("Body length (words)")
ax[1].hist(d.sum_w.clip(upper=d.sum_w.quantile(.99)),  bins=60); ax[1].set_title("Summary length (words)")
ax[2].hist(d.ratio, bins=60);                                    ax[2].set_title("Summary / body ratio")
for a in ax: a.set_ylabel("articles")
plt.tight_layout(); plt.savefig(REP/"length_hist.png", dpi=150); plt.show()

## 7. ข้อสังเกตสำหรับทีม (เติมหลังรัน)
- ค่า percentile 95/99 ของ body → ใช้กำหนด `max_source_length` ของโมเดล (เช่น mT5)
- ค่า percentile 95/99 ของ summary → กำหนด `max_target_length` / `max_new_tokens`
- หมายเหตุ: ความยาวเป็น **คำ (newmm)** ไม่ใช่ **subword token** ของโมเดล ควรนับซ้ำด้วย tokenizer จริงของโมเดลก่อนตั้ง max length